# 통합 및 전처리 단계

In [8]:
import pandas as pd

# 파일 불러오기
a_2021 = pd.read_csv('data/화재사고 구조출동 현황/화재사고_2021.csv')
a_2022 = pd.read_csv('data/화재사고 구조출동 현황/화재사고_2022_preprocessed.csv')
a_2023 = pd.read_csv('data/화재사고 구조출동 현황/화재사고_2023_preprocessed.csv')
a_2024 = pd.read_csv('data/화재사고 구조출동 현황/화재사고_2024.csv')
# 데이터 통합
a_all = pd.concat([a_2021, a_2022, a_2023, a_2024], ignore_index=True)

# 결측치 확인
missing_a = a_all.isnull().sum()

print("화재사고 구조출동 dataset 결측치:")
print(missing_a)

missing_ratio = a_all.isnull().mean() * 100
print("결측치 비율(%):")
print(missing_ratio)

화재사고 구조출동 dataset 결측치:
CLMTY_RSCU_RPTP_NO           0
ACDNT_CS_NM                  0
PRCS_RSLT_SE_NM              0
DCLR_YMD                     0
DCLR_TM                      0
DCLR_YR                      0
SEASN_NM                     0
QTR_NO                       0
DCLR_MM                      0
DCLR_DAY                     0
DCLR_HR                      0
DCLR_MN                      0
DCLR_DOW                     0
DSPT_YMD                  4001
DSPT_TM                   4001
DSPT_YR                   4001
DSPT_MM                   4001
DSPT_DAY                  4001
DSPT_HR                   4001
DSPT_MN                   4001
GRNDS_ARVL_YMD           12076
GRNDS_ARVL_TM            12076
GRNDS_ARVL_YR            12076
GRNDS_ARVL_MM            12076
GRNDS_ARVL_DAY           12076
GRNDS_ARVL_HR            12076
GRNDS_ARVL_MN            12076
RSCU_CMPTN_YMD           22441
RSCU_CMPTN_TM            22441
RSCU_CMPTN_YR            22440
RSCU_CMPTN_MM            22442
RSCU_CMPTN_DAY  

In [9]:
# 컬럼 제거
cols_to_drop = [
    'QTR_NO',
    'LFDAU_NM',
    'HR_UNIT_RN',
    'HR_UNIT_SNWFL',
    'HR_UNIT_VSDST',
    'HR_UNIT_ARTMP',
    'HR_UNIT_HUM',
    'HR_UNIT_WSPD',
    'HR_UNIT_WNDRCT',
    'CBK_YMD',
    'CBK_TM',
    'CBK_YR',
    'CBK_MM',
    'CBK_DAY',
    'CBK_HR',
    'CBK_MN',
    'ETC_OCRN_TYPE_DTL_NM',
    'RSCU_CMPTN_YMD',
    'RSCU_CMPTN_TM',
    'RSCU_CMPTN_YR', 
    'RSCU_CMPTN_MM',
    'RSCU_CMPTN_DAY',
    'RSCU_CMPTN_HR',
    'RSCU_CMPTN_MN' 
]

a_all = a_all.drop(columns=cols_to_drop)

# row 제거
a_all = a_all.dropna(subset=['ACDNT_OCRN_PLC_NM', 'DAMG_RGN_LOT', 'DAMG_RGN_LAT'])

In [10]:
missing_ratio = a_all.isnull().mean() * 100
print("전처리 후 결측치 비율(%):")
print(missing_ratio)

전처리 후 결측치 비율(%):
CLMTY_RSCU_RPTP_NO     0.000000
ACDNT_CS_NM            0.000000
PRCS_RSLT_SE_NM        0.000000
DCLR_YMD               0.000000
DCLR_TM                0.000000
DCLR_YR                0.000000
SEASN_NM               0.000000
DCLR_MM                0.000000
DCLR_DAY               0.000000
DCLR_HR                0.000000
DCLR_MN                0.000000
DCLR_DOW               0.000000
DSPT_YMD               1.112240
DSPT_TM                1.112240
DSPT_YR                1.112240
DSPT_MM                1.112240
DSPT_DAY               1.112240
DSPT_HR                1.112240
DSPT_MN                1.112240
GRNDS_ARVL_YMD         3.356263
GRNDS_ARVL_TM          3.356263
GRNDS_ARVL_YR          3.356263
GRNDS_ARVL_MM          3.356263
GRNDS_ARVL_DAY         3.356263
GRNDS_ARVL_HR          3.356263
GRNDS_ARVL_MN          3.356263
GRNDS_CTPV_NM          0.000838
GRNDS_SGG_NM           0.000838
CTY_FRMVL_SE_NM        0.000000
DAMG_RGN_LOT           0.000000
DAMG_RGN_LAT           

In [11]:
import pandas as pd
import math
# 특수처리: grnds_dstnc(현장거리)
# 1️⃣ 안전센터 좌표 딕셔너리 생성
center_df = pd.read_csv('data/위치관련/서울시 안전센터 위치정보_preprocessed.csv')

center_dict = {
    row['서ㆍ센터명']: (row['longitude'], row['latitude'])
    for _, row in center_df.iterrows()
}

# 2️⃣ haversine 함수 (소수점 1자리 반올림)
def haversine(lon1, lat1, lon2, lat2):
    R = 6371.0
    
    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return round(R * c, 1)

# 3️⃣ 결측치 처리
filled_indices = []

drop_idx = []

for idx, row in a_all.iterrows():
    if pd.isnull(row['GRNDS_DSTNC']):
        cntr_nm = str(row['CNTR_NM'])
        
        # 안전센터 아닌 경우 → 제거 대상
        if '119안전센터' not in cntr_nm:
            drop_idx.append(idx)
        else:
            # 센터명 추출 (딕셔너리 매칭용)
            matched_center = None
            for key in center_dict.keys():
                if key in cntr_nm:
                    matched_center = key
                    break
            
            if matched_center:
                center_lon, center_lat = center_dict[matched_center]
                lon = row['DAMG_RGN_LOT']
                lat = row['DAMG_RGN_LAT']
                
                dist = haversine(lon, lat, center_lon, center_lat)
                a_all.at[idx, 'GRNDS_DSTNC'] = dist
                filled_indices.append(idx)
            else:
                drop_idx.append(idx)

# 실제 제거
a_all = a_all.drop(index=drop_idx)
# 모든 컬럼 보이게 설정
pd.set_option('display.max_columns', None)

# 컬럼 너비 제한 해제 (잘림 방지)
pd.set_option('display.width', None)

# 한 컬럼 내용도 길게 보기
pd.set_option('display.max_colwidth', None)

# 다시 출력
print(a_all.loc[filled_indices].head())

   CLMTY_RSCU_RPTP_NO ACDNT_CS_NM PRCS_RSLT_SE_NM    DCLR_YMD  DCLR_TM  \
3   20171115105S00001          화재            안전조치  20170101.0     39.0   
4   20171115106S00094          화재            안전조치  20170101.0     39.0   
8   20171109102S00003          화재            안전조치  20170101.0   1035.0   
10  20171115106S00003          화재            안전조치  20170101.0  30910.0   
13  20171115104S00002          화재            안전조치  20170101.0  33012.0   

    DCLR_YR SEASN_NM  DCLR_MM  DCLR_DAY  DCLR_HR  DCLR_MN DCLR_DOW  \
3    2017.0       겨울      1.0       1.0      0.0      0.0      일요일   
4    2017.0       겨울      1.0       1.0      0.0      0.0      일요일   
8    2017.0       겨울      1.0       1.0      0.0     10.0      일요일   
10   2017.0       겨울      1.0       1.0      3.0      9.0      일요일   
13   2017.0       겨울      1.0       1.0      3.0     30.0      일요일   

      DSPT_YMD  DSPT_TM  DSPT_YR  DSPT_MM  DSPT_DAY  DSPT_HR  DSPT_MN  \
3   20170101.0    100.0   2017.0      1.0       1.0      0.0 

## 최종 전처리 후 비교

In [12]:
# 결측치 확인
missing_a = a_all.isnull().sum()

print("전처리 후 화재사고 구조출동 dataset 결측치:")
print(missing_a)

전처리 후 화재사고 구조출동 dataset 결측치:
CLMTY_RSCU_RPTP_NO        0
ACDNT_CS_NM               0
PRCS_RSLT_SE_NM           0
DCLR_YMD                  0
DCLR_TM                   0
DCLR_YR                   0
SEASN_NM                  0
DCLR_MM                   0
DCLR_DAY                  0
DCLR_HR                   0
DCLR_MN                   0
DCLR_DOW                  0
DSPT_YMD               2858
DSPT_TM                2858
DSPT_YR                2858
DSPT_MM                2858
DSPT_DAY               2858
DSPT_HR                2858
DSPT_MN                2858
GRNDS_ARVL_YMD        10037
GRNDS_ARVL_TM         10037
GRNDS_ARVL_YR         10037
GRNDS_ARVL_MM         10037
GRNDS_ARVL_DAY        10037
GRNDS_ARVL_HR         10037
GRNDS_ARVL_MN         10037
GRNDS_CTPV_NM             3
GRNDS_SGG_NM              3
CTY_FRMVL_SE_NM           0
DAMG_RGN_LOT              0
DAMG_RGN_LAT              0
GRNDS_DSTNC               0
ACDNT_OCRN_PLC_NM         0
ACDNT_CS_ASSRT_NM         0
FRSTN_NM           

In [13]:
missing_ratio = a_all.isnull().mean() * 100
print("전처리 후 결측치 비율(%):")
print(missing_ratio)

전처리 후 결측치 비율(%):
CLMTY_RSCU_RPTP_NO    0.000000
ACDNT_CS_NM           0.000000
PRCS_RSLT_SE_NM       0.000000
DCLR_YMD              0.000000
DCLR_TM               0.000000
DCLR_YR               0.000000
SEASN_NM              0.000000
DCLR_MM               0.000000
DCLR_DAY              0.000000
DCLR_HR               0.000000
DCLR_MN               0.000000
DCLR_DOW              0.000000
DSPT_YMD              1.041644
DSPT_TM               1.041644
DSPT_YR               1.041644
DSPT_MM               1.041644
DSPT_DAY              1.041644
DSPT_HR               1.041644
DSPT_MN               1.041644
GRNDS_ARVL_YMD        3.658145
GRNDS_ARVL_TM         3.658145
GRNDS_ARVL_YR         3.658145
GRNDS_ARVL_MM         3.658145
GRNDS_ARVL_DAY        3.658145
GRNDS_ARVL_HR         3.658145
GRNDS_ARVL_MN         3.658145
GRNDS_CTPV_NM         0.001093
GRNDS_SGG_NM          0.001093
CTY_FRMVL_SE_NM       0.000000
DAMG_RGN_LOT          0.000000
DAMG_RGN_LAT          0.000000
GRNDS_DSTNC           

In [14]:
a_all.to_csv('data/화재사고 구조출동 현황/화재사고 구조출동 현황(2021-2024)_preprocessed.csv', index=False)

## 나머지 구조출동 현황은 ipynb 별로.